# Week 2 — Flower Classifier & House Price Predictor

**Theme:** Supervised learning I — k-nearest neighbors & linear regression

Supervised learning means: we have labeled examples (input -> correct answer),
and we want the computer to learn the pattern well enough to predict the answer
for *new*, unseen inputs. There are two flavors:

- **Classification** — the answer is a category (e.g. which species of flower,
  benign vs. malignant)
- **Regression** — the answer is a number (e.g. a price, a score)

**k-Nearest Neighbors (k-NN)** is the simplest way to do either: to predict a
new example, find its `k` closest neighbors among the examples we already
know the answer for, then **vote** (classification) or **average**
(regression). We'll try it on three datasets — two classification, one
regression — then move on to linear regression.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris, load_breast_cancer, fetch_california_housing, load_diabetes, load_digits
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso, RidgeCV, LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, mean_squared_error, r2_score

## Part A — k-Nearest Neighbors (Classification & Regression)

세 가지 데이터셋으로 연습합니다: **Iris**(품종 분류), **Breast Cancer**(양성/
악성 분류), **California Housing**(주택 가격 회귀). 먼저 각각 `k=1`로 기본
흐름 — **데이터 나누기 → 모델 만들기(메서드 호출) → 학습 → 평가** — 을
살펴봅니다.

### 1. Iris — 꽃 품종 분류

150개 꽃, 4개 측정값(꽃받침/꽃잎 길이·너비), 3개 품종.

In [ ]:
iris = load_iris()
X_iris, y_iris = iris.data, iris.target
print("Features:", iris.feature_names)
print("Species:", iris.target_names)
print("Shape:", X_iris.shape)

In [ ]:
# 데이터 나누기: 학습용 vs 테스트용 (테스트 데이터는 절대 학습에 쓰지 않습니다)
X_iris_train, X_iris_test, y_iris_train, y_iris_test = train_test_split(
    X_iris, y_iris, test_size=0.3, random_state=42, stratify=y_iris
)

# 모델 만들기(메서드 호출) -> 학습 -> 평가, 우선 k=1부터 시작해봅니다
knn_iris = KNeighborsClassifier(n_neighbors=1)
knn_iris.fit(X_iris_train, y_iris_train)

predictions = knn_iris.predict(X_iris_test)
accuracy = accuracy_score(y_iris_test, predictions)
print(f"Test accuracy with k=1: {accuracy:.2%}")

**참고:** 아래는 4개 측정값 중 2개(꽃잎 길이/너비)만 써서 k-NN이 실제로
공간을 어떻게 나누는지 시각화한 것입니다 (부드러운 경계를 보여주기 위해
`k=5`로 그렸습니다 — 위 `k=1`보다 경계가 훨씬 덜 들쭉날쭉합니다).

In [ ]:
X_iris2 = X_iris[:, 2:4]  # petal length, petal width
X_iris2_train, X_iris2_test, y_iris2_train, y_iris2_test = train_test_split(
    X_iris2, y_iris, test_size=0.3, random_state=42, stratify=y_iris
)
knn_iris2 = KNeighborsClassifier(n_neighbors=5).fit(X_iris2_train, y_iris2_train)

xx, yy = np.meshgrid(
    np.linspace(X_iris2[:, 0].min() - 0.5, X_iris2[:, 0].max() + 0.5, 200),
    np.linspace(X_iris2[:, 1].min() - 0.5, X_iris2[:, 1].max() + 0.5, 200),
)
Z = knn_iris2.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

plt.figure(figsize=(6, 5))
plt.contourf(xx, yy, Z, alpha=0.25, cmap="viridis")
plt.scatter(X_iris2[:, 0], X_iris2[:, 1], c=y_iris, cmap="viridis", edgecolor="k")
plt.title("k-NN Decision Boundary (petal length vs. width, k=5)")
plt.xlabel(iris.feature_names[2])
plt.ylabel(iris.feature_names[3])
plt.show()

### 2. Breast Cancer — 양성/악성 분류

569개 종양 샘플, 30개 측정값(세포 크기, 질감 등), 2개 클래스
(malignant/benign).

In [ ]:
cancer = load_breast_cancer()
X_cancer, y_cancer = cancer.data, cancer.target
print("Features:", len(cancer.feature_names))
print("Classes:", list(cancer.target_names))
print("Shape:", X_cancer.shape)

In [ ]:
X_cancer_train, X_cancer_test, y_cancer_train, y_cancer_test = train_test_split(
    X_cancer, y_cancer, test_size=0.3, random_state=42, stratify=y_cancer
)

knn_cancer = KNeighborsClassifier(n_neighbors=1)
knn_cancer.fit(X_cancer_train, y_cancer_train)

predictions = knn_cancer.predict(X_cancer_test)
accuracy = accuracy_score(y_cancer_test, predictions)
print(f"Test accuracy with k=1: {accuracy:.2%}")

### 3. California Housing — 주택 가격 회귀

k-NN은 회귀에도 쓸 수 있습니다 — 다수결 투표 대신, 가장 가까운 `k`개
이웃의 **타깃값을 평균**냅니다. 20,640개 캘리포니아 지역구, 8개 특징
(소득, 방 개수, 위치 등)으로 중간 주택 가격(단위: $100,000)을 예측합니다.

*(처음 실행할 때 데이터를 내려받느라 몇 초 걸릴 수 있습니다.)*

In [ ]:
housing = fetch_california_housing()
X_housing, y_housing = housing.data, housing.target
print("Features:", housing.feature_names)
print("Shape:", X_housing.shape)

In [ ]:
X_housing_train, X_housing_test, y_housing_train, y_housing_test = train_test_split(
    X_housing, y_housing, test_size=0.3, random_state=42
)

# 모델 만들기 -> 학습 -> 평가. 분류가 아니라 회귀이므로 Classifier가 아닌
# Regressor를 씁니다 -- 나머지 흐름은 완전히 동일합니다.
knn_housing = KNeighborsRegressor(n_neighbors=1)
knn_housing.fit(X_housing_train, y_housing_train)

predictions = knn_housing.predict(X_housing_test)
print(f"R^2 with k=1:  {r2_score(y_housing_test, predictions):.3f}")
print(f"RMSE with k=1: {mean_squared_error(y_housing_test, predictions) ** 0.5:.3f}  (unit: $100,000)")

## k 값을 바꿔보면 어떻게 될까?

지금까지는 세 데이터셋 모두 `k=1`만 써봤습니다 — 가장 가까운 이웃 **딱
하나**에만 의존하는 거라 노이즈에 민감할 수 있습니다. `k`를 늘리면 성능이
어떻게 달라질까요?

아래 세 칸에서 직접 실험해보세요 — 각 데이터셋에 대해 `k`를 1부터 20까지
바꿔가며 성능(분류는 accuracy, 회귀는 RMSE)을 계산해서 리스트에 저장하고,
`k`에 따라 어떻게 변하는지 그래프로 그려보세요. (힌트: 위에서 `k=1`로 했던
코드를 그대로 가져와서 `for k in range(1, 21):` 반복문 안에 넣고, 매번 새
모델을 만들어 학습·평가한 뒤 결과를 리스트에 추가하면 됩니다.)

In [ ]:
# TODO: Iris에서 k=1~20까지 정확도를 계산해 리스트에 저장하고,
# k에 따른 정확도 변화를 그래프로 그려보세요.


In [ ]:
# TODO: Breast Cancer에서 k=1~20까지 정확도를 계산해 리스트에 저장하고,
# k에 따른 정확도 변화를 그래프로 그려보세요.


In [ ]:
# TODO: California Housing에서 k=1~20까지 RMSE를 계산해 리스트에 저장하고,
# k에 따른 RMSE 변화를 그래프로 그려보세요.


## Part B — Regression: Linear Regression & Regularization

**Idea:** fit a straight line (or plane, in higher dimensions) through the data
that best predicts a numeric target. We'll start with plain
`LinearRegression`, then see what happens when there are many features and
how `Ridge`/`Lasso` (and their auto-tuning versions `RidgeCV`/`LassoCV`)
help.

### 1. Linear Regression — diabetes (특징 1개)

The **diabetes** dataset: 442 patients, several health measurements, and a
target that measures disease progression one year later. We'll start with just
one feature — BMI — so we can plot the fitted line directly.

In [ ]:
diabetes = load_diabetes()
bmi = diabetes.data[:, diabetes.feature_names.index("bmi")].reshape(-1, 1)
target = diabetes.target

Xb_train, Xb_test, yb_train, yb_test = train_test_split(
    bmi, target, test_size=0.3, random_state=42
)

reg = LinearRegression()
reg.fit(Xb_train, yb_train)

print(f"Learned line: progression = {reg.coef_[0]:.1f} * bmi + {reg.intercept_:.1f}")

In [ ]:
predictions = reg.predict(Xb_test)
print(f"R^2 score:  {r2_score(yb_test, predictions):.3f}  (1.0 = perfect, 0.0 = no better than guessing the mean)")
print(f"RMSE:       {mean_squared_error(yb_test, predictions) ** 0.5:.1f}")

In [ ]:
plt.figure(figsize=(6, 5))
plt.scatter(Xb_test, yb_test, alpha=0.6, label="actual")
order = np.argsort(Xb_test[:, 0])
plt.plot(Xb_test[order], predictions[order], color="red", linewidth=2, label="predicted line")
plt.title("Linear Regression: BMI -> Disease Progression")
plt.xlabel("BMI (standardized)")
plt.ylabel("Disease progression score")
plt.legend()
plt.show()

### 2. Ridge / Lasso / RidgeCV / LassoCV — Gisette (숫자 4 vs 9, 특징 5000개)

Wine Quality는 특징이 11개뿐이라 정규화가 크게 도움되지 않았죠. 이번엔
**Gisette** 데이터셋을 씁니다 — 손글씨 숫자 **4와 9를 구분**하는 문제인데,
사람 눈에도 헷갈리는 두 숫자를 일부러 골랐고, 특징을 무려 **5,000개**나
줍니다. 그런데 이 중 **절반 정도(약 2,500개)는 예측에 전혀 도움이 안 되는
순수 노이즈 특징**입니다 (2003년 NIPS 특징 선택 대회를 위해 일부러 이렇게
설계된 데이터셋입니다). 게다가 학습 데이터 행 수(약 5,000개)가 특징 개수와
비슷한 수준이라, 정규화 없는 일반 회귀는 심하게 과적합하기 쉬운 조건입니다
— Ridge/Lasso가 확실히 도움되는 걸 보기에 훨씬 좋은 예시입니다.

**참고용 이미지:** 아래는 Gisette 데이터 자체가 아니라, sklearn에 내장된
작은 손글씨 숫자 샘플(`load_digits`, 인터넷 연결 불필요)에서 뽑은 실제
4와 9입니다 — 두 숫자가 왜 헷갈리는지 감을 잡는 용도입니다. (Gisette의
5,000개 특징은 픽셀을 그대로 나열한 것이 아니라, 여러 픽셀을 조합/변형하고
노이즈 특징까지 섞어 놓은 것이라 이미지로 바로 그릴 수는 없습니다.)

In [ ]:
digits = load_digits()
fours = digits.images[digits.target == 4]
nines = digits.images[digits.target == 9]

fig, axes = plt.subplots(2, 5, figsize=(8, 3.5))
for ax, img in zip(axes[0], fours[:5]):
    ax.imshow(img, cmap="gray_r")
    ax.set_title("4", fontsize=10)
    ax.axis("off")
for ax, img in zip(axes[1], nines[:5]):
    ax.imshow(img, cmap="gray_r")
    ax.set_title("9", fontsize=10)
    ax.axis("off")
plt.suptitle("Examples of the digits 4 and 9 (sklearn load_digits, not Gisette itself)")
plt.tight_layout()
plt.show()

In [ ]:
!pip install -q kagglehub
import kagglehub
import glob
import pickle

gisette_dir = kagglehub.dataset_download("fedesoriano/gisette-dataset-mnist-digits-4-and-9")
pkl_candidates = glob.glob(f"{gisette_dir}/**/*.pkl", recursive=True) + \
                 glob.glob(f"{gisette_dir}/**/*.pickle", recursive=True)
pkl_path = pkl_candidates[0]
print("Using file:", pkl_path)

with open(pkl_path, "rb") as f:
    gisette_data = pickle.load(f)

print("Top-level keys:", list(gisette_data.keys()))

In [ ]:
# training + validation 세트를 합쳐서 우리만의 train/test로 다시 나눕니다
# (Gisette 원본의 test 세트는 label이 공개되지 않은 벤치마크용이라 사용하지 않습니다)
# float64로 변환합니다 (float32는 특징이 샘플보다 많은 경우 Ridge의
# 내부 계산이 수치적으로 불안정해질 수 있습니다)
X_gisette = np.vstack([gisette_data["training"]["data"], gisette_data["validation"]["data"]]).astype(np.float64)
y_raw = np.concatenate([gisette_data["training"]["labels"], gisette_data["validation"]["labels"]])

raw_classes = sorted(set(y_raw.tolist()))
print("Raw label values found:", raw_classes)
assert len(raw_classes) == 2, f"expected 2 classes, got {raw_classes}"

# 실제 값이 무엇이든 -1 / +1로 표준화 (부호로 클래스를 구분하기 위함)
y_gisette = np.where(y_raw == raw_classes[0], -1, 1)

print("Shape:", X_gisette.shape)
print("Class balance:", pd.Series(y_gisette).value_counts().to_dict())

In [ ]:
X_gisette_train, X_gisette_test, y_gisette_train, y_gisette_test = train_test_split(
    X_gisette, y_gisette, test_size=0.3, random_state=42, stratify=y_gisette
)
print("Train:", X_gisette_train.shape, "Test:", X_gisette_test.shape)

#### 특징 스케일링

Wine Quality 때와 같은 이유로, Ridge/Lasso를 쓰기 전에 `StandardScaler`로
모든 특징을 평균 0, 표준편차 1로 맞춥니다 (학습 데이터에만 `fit`).

In [ ]:
scaler_g = StandardScaler()
X_gisette_train_scaled = scaler_g.fit_transform(X_gisette_train)
X_gisette_test_scaled = scaler_g.transform(X_gisette_test)

#### 회귀 모델로 분류하기

지금 타깃(`y_gisette`)은 숫자가 아니라 클래스(-1 또는 +1)입니다. 그런데
회귀 모델(`LinearRegression`, `Ridge`, `Lasso`, ...)은 원래 연속적인 숫자를
예측하는 모델이죠. 이럴 때 흔히 쓰는 트릭이 있습니다: 타깃을 그냥 -1과
+1이라는 **숫자**로 두고 회귀로 예측한 뒤, **예측값의 부호(양수면 +1,
음수면 -1)** 로 클래스를 정하는 것입니다. 사실 sklearn의 `RidgeClassifier`가
내부적으로 정확히 이렇게 동작합니다 — 우리는 그 과정을 직접 손으로
해보는 셈입니다. 그래서 아래부터는 RMSE 대신 **분류 정확도(accuracy)**를
주로 봅니다.

#### (1) Linear Regression — 기준선

대표적인 인자:

- **`fit_intercept`** (기본값 `True`) — 절편을 포함할지 여부.
- **`positive`** (기본값 `False`) — `True`면 모든 계수를 양수로 제한.
- **`n_jobs`** (기본값 `None`) — 계산에 사용할 CPU 코어 수.

특징이 5,000개인데 학습 데이터는 약 4,900개뿐입니다 — 특징 수가 샘플
수에 육박하면 정규화 없는 회귀는 학습 데이터에는 거의 완벽히 맞아도
(overfitting) 테스트 데이터에서는 성능이 크게 떨어지기 쉽습니다. 실제로
그런지 확인해봅시다.

In [ ]:
lr_gisette = LinearRegression()
lr_gisette.fit(X_gisette_train_scaled, y_gisette_train)

train_preds_lr = lr_gisette.predict(X_gisette_train_scaled)
test_preds_lr = lr_gisette.predict(X_gisette_test_scaled)
train_acc_lr = (np.sign(train_preds_lr) == y_gisette_train).mean()
test_acc_lr = (np.sign(test_preds_lr) == y_gisette_test).mean()

print(f"[Linear Regression] Train accuracy: {train_acc_lr:.2%}")
print(f"[Linear Regression] Test accuracy:  {test_acc_lr:.2%}")
print(f"[Linear Regression] Test RMSE:       {mean_squared_error(y_gisette_test, test_preds_lr) ** 0.5:.3f}")

#### (2) Ridge Regression — L2 정규화

대표적인 인자:

- **`alpha`** (기본값 `1.0`) — 계수 크기에 대한 페널티 강도. 클수록
  계수들이 0 쪽으로 눌려서 모델이 단순해지고 과적합이 줄지만, 너무 크면
  과소적합이 됩니다. 보통 `0.01, 0.1, 1.0, 10.0, 100.0`처럼 로그 스케일로
  후보를 고릅니다.
- **`solver`** (기본값 `'auto'`) — 계수를 계산하는 방법. 후보:
  `'auto', 'svd', 'cholesky', 'lsqr', 'sparse_cg', 'sag', 'saga', 'lbfgs'`.
- **`max_iter`** (기본값 `None`) — 반복적인 solver를 쓸 때 최대 반복 횟수.

지금은 `alpha=1.0`(기본값)을 그대로 써봅니다.

In [ ]:
ridge_gisette = Ridge(alpha=1.0)
ridge_gisette.fit(X_gisette_train_scaled, y_gisette_train)

test_preds_ridge = ridge_gisette.predict(X_gisette_test_scaled)
test_acc_ridge = (np.sign(test_preds_ridge) == y_gisette_test).mean()

print(f"[Ridge, alpha=1.0] Test accuracy: {test_acc_ridge:.2%}")
print(f"[Ridge, alpha=1.0] Test RMSE:     {mean_squared_error(y_gisette_test, test_preds_ridge) ** 0.5:.3f}")

#### (3) Lasso Regression — L1 정규화

대표적인 인자:

- **`alpha`** (기본값 `1.0`) — Ridge와 같은 역할이지만, L1 페널티는 덜
  중요한 특징의 계수를 아예 **0으로** 만듭니다 — 자동 특징 선택 효과.
  Lasso는 alpha가 조금만 커져도 계수가 급격히 0이 되는 경우가 많아서
  보통 `0.001, 0.01, 0.1, 1.0`처럼 더 작은 값부터 후보를 고릅니다.
- **`max_iter`** (기본값 `1000`) — 좌표하강법 반복의 최대 횟수. 특징이
  5,000개나 되면 기본값으로는 수렴 전에 반복이 끝날 수 있어 늘려줍니다.
- **`selection`** (기본값 `'cyclic'`) — 계수를 업데이트할 순서. 후보:
  `'cyclic'`, `'random'`.

Gisette는 특징 5,000개 중 절반이 순수 노이즈라고 했죠 — Lasso가 그
노이즈 특징들의 계수를 실제로 0으로 만드는지 확인해봅니다.

In [ ]:
lasso_gisette = Lasso(alpha=0.01, max_iter=5000)
lasso_gisette.fit(X_gisette_train_scaled, y_gisette_train)

test_preds_lasso = lasso_gisette.predict(X_gisette_test_scaled)
test_acc_lasso = (np.sign(test_preds_lasso) == y_gisette_test).mean()
n_zero_lasso = int((lasso_gisette.coef_ == 0).sum())

print(f"[Lasso, alpha=0.01] Test accuracy: {test_acc_lasso:.2%}")
print(f"[Lasso, alpha=0.01] Test RMSE:     {mean_squared_error(y_gisette_test, test_preds_lasso) ** 0.5:.3f}")
print(f"0으로 눌린 계수 개수: {n_zero_lasso} / {len(lasso_gisette.coef_)}")

#### alpha, 매번 손으로 골라야 할까?

방금까지는 `alpha`를 하나 정해서(`1.0`, `0.01`) 넣어봤습니다. 어떤 값이
가장 좋은지는 미리 알 수 없으니, 여러 후보를 다 시도해보고 검증 성능이
가장 좋은 걸 고르는 게 최선입니다. **`RidgeCV`/`LassoCV`**는 이 과정(여러
alpha 후보 × 교차검증)을 자동으로 해주는 버전입니다.

#### (4) RidgeCV

대표적인 인자:

- **`alphas`** (기본값 `(0.1, 1.0, 10.0)`) — 시도해볼 alpha 후보 리스트.
  보통 `np.logspace(-3, 3, 13)`처럼 더 넓게 지정하지만, 여기서는 특징이 5,000개라 alpha 후보 하나당 학습이 오래 걸리므로 `[0.1, 1, 10, 100]`으로 후보를 줄여서 실습합니다.
- **`cv`** (기본값 `None`) — `None`이면 효율적인 **Leave-One-Out**
  교차검증을 사용합니다. 정수(예: `cv=5`)를 주면 일반 k-fold로 바뀝니다.
- **`scoring`** (기본값 `None`) — 각 alpha를 평가할 지표. `None`이면 R².

학습이 끝나면 `.alpha_`에 선택된 최적 alpha가 저장됩니다.

*(특징이 샘플보다 많은 데이터에서 아주 작은 alpha를 시도할 때 `Singular matrix in solving dual problem` 이라는 경고가 뜰 수 있는데, 정규화가 거의 없어 계산이 수치적으로 불안정해지는 경우 sklearn이 자동으로 다른 방식(최소제곱해)으로 대체 계산했다는 뜻입니다 -- 학습은 정상적으로 끝나니 무시해도 됩니다.)

In [ ]:
ridge_cv_gisette = RidgeCV(alphas=[0.1, 1, 10, 100], cv=5)
ridge_cv_gisette.fit(X_gisette_train_scaled, y_gisette_train)

test_preds_ridge_cv = ridge_cv_gisette.predict(X_gisette_test_scaled)
test_acc_ridge_cv = (np.sign(test_preds_ridge_cv) == y_gisette_test).mean()

print(f"[RidgeCV] selected alpha: {ridge_cv_gisette.alpha_:.4f}")
print(f"[RidgeCV] Test accuracy: {test_acc_ridge_cv:.2%}")
print(f"[RidgeCV] Test RMSE:     {mean_squared_error(y_gisette_test, test_preds_ridge_cv) ** 0.5:.3f}")

#### (5) LassoCV

대표적인 인자:

- **`alphas`** (기본값 `100`) — 정수를 주면 그 개수만큼 alpha 후보를
  자동으로 생성합니다 (직접 리스트를 줄 수도 있습니다).
- **`cv`** (기본값 `None`) — `None`이면 **5-fold** 교차검증 (RidgeCV의
  `None`과 의미가 다릅니다!).
- **`eps`** (기본값 `0.001`) — 후보 alpha의 최소/최대 비율.
- **`n_jobs`** — 병렬 처리에 사용할 CPU 코어 수. 특징이 5,000개라 계산이
  꽤 걸리므로 `-1`(가능한 모든 코어)로 지정합니다.

*(alpha 후보를 4개로 줄였지만, 특징이 5,000개라 여전히 앞의 셀들보다는 실행이 오래 걸릴 수 있습니다.)*

In [ ]:
lasso_cv_gisette = LassoCV(alphas=[0.1, 1, 10, 100], cv=5, random_state=42, n_jobs=-1, max_iter=5000)
lasso_cv_gisette.fit(X_gisette_train_scaled, y_gisette_train)

test_preds_lasso_cv = lasso_cv_gisette.predict(X_gisette_test_scaled)
test_acc_lasso_cv = (np.sign(test_preds_lasso_cv) == y_gisette_test).mean()
n_zero_lasso_cv = int((lasso_cv_gisette.coef_ == 0).sum())

print(f"[LassoCV] selected alpha: {lasso_cv_gisette.alpha_:.4f}")
print(f"[LassoCV] Test accuracy: {test_acc_lasso_cv:.2%}")
print(f"[LassoCV] Test RMSE:     {mean_squared_error(y_gisette_test, test_preds_lasso_cv) ** 0.5:.3f}")
print(f"0으로 눌린 계수 개수: {n_zero_lasso_cv} / {len(lasso_cv_gisette.coef_)}")

#### 다섯 모델 비교

정규화가 없을 때(LinearRegression), alpha를 손으로 고정했을 때(Ridge/
Lasso), 자동으로 탐색했을 때(RidgeCV/LassoCV)의 **테스트 정확도**를
한눈에 비교합니다. 노이즈 특징이 절반인 이 데이터에서는 Lasso 계열이
불필요한 특징을 걸러내면서 확실한 차이를 보여줄 가능성이 높습니다.

In [ ]:
gisette_results = pd.DataFrame([
    {"Model": "Linear Regression", "alpha": "-", "Accuracy": test_acc_lr,
     "RMSE": mean_squared_error(y_gisette_test, test_preds_lr) ** 0.5},
    {"Model": "Ridge (fixed alpha)", "alpha": 1.0, "Accuracy": test_acc_ridge,
     "RMSE": mean_squared_error(y_gisette_test, test_preds_ridge) ** 0.5},
    {"Model": "Lasso (fixed alpha)", "alpha": 0.01, "Accuracy": test_acc_lasso,
     "RMSE": mean_squared_error(y_gisette_test, test_preds_lasso) ** 0.5},
    {"Model": "RidgeCV", "alpha": round(ridge_cv_gisette.alpha_, 4), "Accuracy": test_acc_ridge_cv,
     "RMSE": mean_squared_error(y_gisette_test, test_preds_ridge_cv) ** 0.5},
    {"Model": "LassoCV", "alpha": round(lasso_cv_gisette.alpha_, 4), "Accuracy": test_acc_lasso_cv,
     "RMSE": mean_squared_error(y_gisette_test, test_preds_lasso_cv) ** 0.5},
])
display(gisette_results)

plt.figure(figsize=(7, 4))
plt.bar(gisette_results["Model"], gisette_results["Accuracy"], color="steelblue")
plt.title("Gisette (4 vs 9): Test Accuracy by Model")
plt.ylabel("Accuracy")
plt.ylim(0, 1)
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

## Try it yourself

1. **Compare k-NN to a "dumb" baseline.** For Breast Cancer, what accuracy
   would you get by always predicting the majority class? Is k-NN actually
   learning something?
2. **Compare to a "dumb" baseline.** What R² would you get if you always
   predicted the *average* disease progression, regardless of BMI? (Hint:
   that's what an R² of 0 means.)
3. **Add a second feature to the regression.** Use both `bmi` and `s5`
   (another column in `diabetes.feature_names`) as inputs to
   `LinearRegression` — does the R² improve? 아래 셀에 두 특징을 합쳐서
   학습/테스트 데이터를 준비하는 코드를 미리 작성해뒀습니다 — 이어서
   `LinearRegression`으로 학습시키고 R²를 계산해서 비교해보세요.

In [ ]:
# bmi와 s5, 두 특징을 하나의 입력 데이터로 합칩니다
s5 = diabetes.data[:, diabetes.feature_names.index("s5")].reshape(-1, 1)
X_two = np.hstack([bmi, s5])  # (442, 1)짜리 두 배열을 옆으로 이어붙여 (442, 2) 데이터를 만듭니다

Xb2_train, Xb2_test, yb2_train, yb2_test = train_test_split(
    X_two, target, test_size=0.3, random_state=42
)

# TODO: 위에서 나눈 Xb2_train/yb2_train으로 LinearRegression을 학습시키고,
# Xb2_test에 대한 예측의 R² 점수를 계산해서 위 bmi 단일 특징 모델과 비교해보세요.


---
## 🎯 캡스톤: 이번 학기 성적 위험도 예측기

가상의 선배 150명의 "주당 공부시간 / 출석률 / 평균 수면시간 -> 기말 점수" 기록을 드립니다. 이 데이터로 **k-NN 분류기**(위험군 Safe/Warning/Danger 예측)와 **선형회귀**(예상 점수 예측)를 직접 만들어보고, 마지막엔 **여러분 자신의 예상 습관**을 입력해서 결과를 확인해보세요.

**확장 아이디어:** 학기 말에 실제 본인의 공부시간/출석/수면 기록과 실제 성적을 몇 학기치 모아서 `students_df`를 바꿔치기하면, 진짜 "내 성적 예측기"가 됩니다.

In [ ]:
# 더미 데이터 생성 (실행만 하면 됩니다)
import pandas as pd
rng = np.random.default_rng(7)
n_students = 150

weekly_study_hours = np.clip(rng.normal(10, 4, n_students), 0, 25)
attendance_rate = np.clip(rng.normal(0.85, 0.12, n_students), 0.4, 1.0)
sleep_hours_avg = np.clip(rng.normal(6.5, 1.2, n_students), 3, 10)

final_score = (
    20
    + 2.2 * weekly_study_hours
    + 45 * attendance_rate
    + 1.5 * sleep_hours_avg
    + rng.normal(0, 6, n_students)
)
final_score = np.clip(final_score, 0, 100)

def to_risk(score):
    if score >= 80:
        return "Safe"
    elif score >= 60:
        return "Warning"
    return "Danger"

students_df = pd.DataFrame({
    "weekly_study_hours": weekly_study_hours.round(1),
    "attendance_rate": attendance_rate.round(2),
    "sleep_hours_avg": sleep_hours_avg.round(1),
    "final_score": final_score.round(1),
})
students_df["risk"] = students_df["final_score"].apply(to_risk)
students_df.head()

### 여러분의 과제

1. `students_df`에서 `weekly_study_hours`, `attendance_rate`, `sleep_hours_avg` 3개를 입력(X)으로, `risk`를 정답(y)으로 하여 **k-NN 분류기**를 학습시키고 테스트 정확도를 출력하세요. (Part A 코드를 참고하세요: `train_test_split`, `KNeighborsClassifier`, `accuracy_score`)
2. 같은 3개 입력으로 `final_score`(숫자)를 예측하는 **선형회귀 모델**을 학습시키고 R² 점수를 출력하세요. (Part B 코드 참고: `LinearRegression`, `r2_score`)
3. 아래에 **여러분 자신의 예상 습관**(예상 주당 공부시간, 예상 출석률, 예상 평균 수면시간)을 숫자로 입력하고, 학습된 두 모델로 (a) 위험군과 (b) 예상 점수를 각각 예측해서 출력해보세요.

In [ ]:
# TODO 1: k-NN 분류기로 risk(Safe/Warning/Danger)를 예측하는 모델을 학습하고 테스트 정확도를 출력하세요.


# TODO 2: 선형회귀로 final_score를 예측하는 모델을 학습하고 R^2를 출력하세요.


# TODO 3: 나의 예상 습관을 입력하고, 위 두 모델로 위험군과 예상 점수를 예측해보세요.
my_weekly_study_hours = None   # 예: 8
my_attendance_rate = None      # 예: 0.9
my_sleep_hours_avg = None      # 예: 6.5